# 분산 분석(ANOVA)

- 문제)
    - 주어진데이터는 4종류의 비료를 사용한 식물의 성장에 대한 실험결과다. 
    - 이 실험에서는 비슷한 조건의 식물 40개를 무작위로 10개씩 나누고 화학 비료 A, B, C, D를 일정 기간 사용한 후 성장량을 측정했다. 
    - 성장의 차이가 있는지 유의수준 0.05하에서 검정하시오,

In [21]:
import pandas as pd
df = pd.read_csv('fertilizer.csv')
df.sample(10)

,비료,성장
37,D,9.7
2,A,10.8
6,A,10.9
16,B,12.2
11,B,12.4
26,C,10.8
27,C,11.5
29,C,11.0
28,C,11.4
38,D,9.3


## formula(수식) 작성 문법 정리

### ~ (물결)
- 종속변수(타겟, 정답)와 독립변수(피쳐, 입력)를 구분
- target ~a
    - target : 종속변수
    - a : 독립변수

### + (더하기)
- 여러 독립변수를 모델에 포함
- target ~ a + b + c
    - target : 종속변수
    - a, b, c : 여러 독립 변수들

### C (변수)
- 해당 변수를 범주형(Categorical)으로 명시
- 변수가 이미 문자열이면 자동 인식되지만, 숫자로 코딩된 범주형 변수는 반드시 c()로 감싸야 연속형으로 잘못처리되지 않는다.
- 컬럼명에 띄어쓰기나 특수문자가 있으면 formula문법에서 에러가 난다.
    - 먼저, df.rename(columns={'원래컬럼명' : '새 컬럼명'}) 컬럼명을 수정한 뒤 모델을 만들어야 한다.

In [22]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

model = ols('성장 ~ C(비료)', df).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
C(비료),3.0,43.21875,14.406250,89.126139,1.001838e-16
Residual,36.0,5.81900,0.161639,NaN,NaN


### anova_lm() 출력 표 읽기
- df(자유도, degree of freedom) : 그룹 수 -1(요인의 자유도)
                                 전체 관측치 수 - 그룹 수 (잔차의 자유도)
- sum_sq(제곱합, SS) : 그룹별 또는 잔차의 제곱합(sum of squares)
- mean_sq(평균제곱, MS) : sum_sq를 df로 나눈 값
- F(F-통계량) : 그룹 간 분산 / 그룹 내 분산의 비율
- PR(>F)(p-value) : F-통계량에 대한 유의 확률

- 문제) 
    - 데이터는 네 가지 종류의 나무(A, B, C, D)에 대해 세 가지 종류의 비료(1,2,3)를 사용해 성장률을 조사한 결과다. 
    - 비료 간 및 나무 종류 간의 성장률 차이가 있는지 유의수준 0.05하에서 검정하시오 (단, 독립성, 정규성, 등분산성에 만족한 데이터)

## 이원분산분석(Tow-way ANOVA)
- 독립변수가 2개(예:나무 종류, 비료 종류)일 때 각각의 영향과, 두 변수의 조합(상호작용) 효과까지 분석
- 기본 가정은 일원 분산분석과 동일하게 독립성, 정규성(shapiro-wilk), 등분산성(Levene)으로 한다.

- 주효과 2개 + 상호작용 효과 1개 --> 총 3가지 가설을 동시에 검정
    1. 주효과 - 요인 A (예: 학습 방법)
        - 귀무가설 : 학습 방법에 따라 성적 차이가 없다.
        - 대립가설 : 학습 방법에 따라 성적 차이가 있다.
    2. 주효과 - 요인 B (예 학습 장소)
        - 귀무가설 : 학습 장소에 따라 성적 차이가 없다.
        - 대립가설 : 학습 장소에 따라 성적 차이가 있다.
    3. 상호작용 효과
        - 귀무가설 : A 요인과 B 요인 간에 상호작용이 없다.
        - 대립가설 : A 요인과 B 요인 간에 상호작용이 있다.

- statmodels의 ols와 anova_lm을 사용한다.

    

In [23]:
import pandas as pd
df = pd.read_csv('tree.csv')
df.sample(10)   # 무작위로 10개 보여줌

,나무,비료,성장률
112,D,3,71.602302
88,C,3,60.702398
7,A,1,57.674347
71,C,2,78.380366
105,D,2,72.040509
34,B,1,63.225449
82,C,3,80.778940
62,C,1,48.936650
94,D,1,61.078918
37,B,1,35.403299


In [24]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 성장률 ~ 나무 + 비료 + 나무:비료
model = ols('성장률 ~ 나무 + 비료 + 나무:비료', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

,df,sum_sq,mean_sq,F,PR(>F)
나무,3.0,4783.353938,1594.451313,18.391274,9.016693e-10
비료,1.0,873.322002,873.322002,10.073374,1.942421e-03
나무:비료,3.0,394.801585,131.600528,1.517952,2.137666e-01
Residual,112.0,9709.960792,86.696078,NaN,NaN


In [25]:
model = ols('성장률 ~ C(나무) + C(비료) + C(나무):C(비료)', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

,df,sum_sq,mean_sq,F,PR(>F)
C(나무),3.0,4783.353938,1594.451313,18.855528,6.600012e-10
C(비료),2.0,1127.924259,563.962129,6.669256,1.857612e-03
C(나무):C(비료),6.0,717.520672,119.586779,1.414199,2.157357e-01
Residual,108.0,9132.639448,84.561476,NaN,NaN


In [26]:
# F-통계량에 대한 유의 확률
print(format(6.600012e-10, '.11f'))
print(format(1.857612e-03, '.11f'))
print(format(2.157357e-01, '.11f'))

0.00000000066
0.00185761200
0.21573570000


#### 상호작용 효과를 *로 간단히 
- '나무 + 비료 + 나무:비료' --> '나무*비료'

model = ols('성장률 ~ C(나무)*C(비료)', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

In [27]:
model = ols('성장률 ~ C(나무) * C(비료)', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

,df,sum_sq,mean_sq,F,PR(>F)
C(나무),3.0,4783.353938,1594.451313,18.855528,6.600012e-10
C(비료),2.0,1127.924259,563.962129,6.669256,1.857612e-03
C(나무):C(비료),6.0,717.520672,119.586779,1.414199,2.157357e-01
Residual,108.0,9132.639448,84.561476,NaN,NaN


- 나무 종류 --> 성장률에 유의미한 차이가 있다. p < 0.05
- 비료 종류 --> 비료 종류에 따라 성장률에 유의미한 차이가 있다. p < 0.05
- 나무와 비료 간의 상호작용은 성장률에 유의미한 영향을 주지 않느다. p >= 0.05
- 주효과 2개는 유의미, 상호작용은 유읨하지 않은 패턴이 시험에 자주 출제 됨.

### 레빈 검정(Levene)
- center 파라미터에 따라 mean(Levene 검정) 또는 median(Brown_Forsythe 검정)으로 등분산성을 확인할 수 있다.
    - 문제에서 특별히 명시하지 않으면 기본값을 사용한다.

#### anova_lm(model, typ=숫자)
- typ 파라미터로 제곱합 계산 방식을 조정할 수 있다.
- typ=1(기본값) --> 문제에서 특별한 요구를 하지 않으면 기본값을 사용한다. 모델이 단순한 경우(설계가 균형잡힌 경우)  typ값을 바뀌도 결과가 유사하게 나오는 경우가 많다.
    - 순차적 제곱합 - 변수를 하나씩 추가하며 순서대로 영향을 계산
- typ=2
    - 불순차적 제곱합 - 각 변수가 독립적으로 미치는 영향을 계산
- typ=3
    - 제곱합 - 상호작용까지 고려한 상태에서 각 변수의 효과 계산